# 11 - Gradient Boosting

The third and last planned model, evaluated exactly like the logistic regression (notebook 09) and the random forest (notebook 10), and compared with both. Same protocol: 12 development patients, the same 4 grouped cross-validation folds, the same 13 features. The 5 test patients stay sealed until every model is built and one has been chosen.

## What boosting is, and why this implementation

**Boosting** builds the model one small tree at a time. Each new tree is fitted to the *mistakes* the ensemble has made so far (technically the gradient of the log-loss) and its output is added in, shrunk by a learning rate. The final score is the sum of all trees' contributions, turned into a probability.

Compared with the two models already built:

- A **random forest** averages many independent, deep trees, which reduces *variance*. **Boosting** adds shallow trees one after another, each correcting the last, which reduces *bias* and can fit subtler patterns - but can also chase noise, including patient-specific quirks, if it runs too long.
- **Logistic regression** is one linear surface; boosting is a sum of many small non-linear rules.

**Why `HistGradientBoostingClassifier`.** It is scikit-learn's modern default for tabular data of this size. It first bins each feature into at most 255 histogram bins, which makes it fast (all four cross-validation folds take about a second) and insensitive to feature scale and to the heavy-tailed RR values; it has sensible defaults; and it needs no extra library. The classic `GradientBoostingClassifier` implements the same idea more slowly (it scans every split candidate instead of histogram bins); AdaBoost is an older variant that reweights beats rather than fitting gradients; XGBoost and LightGBM are external packages we do not need.

**Settings: scikit-learn defaults, chosen before looking at any result** - learning rate 0.1, up to 31 leaves per tree, at least 20 beats per leaf, 100 boosting rounds. One deliberate override: `early_stopping` is off. With its `"auto"` default, on more than 10,000 rows scikit-learn holds out a random 10% of the *training beats* as a validation set - mixing patients between fitting and validation inside the training data - and stops when that looks best, which would tune the model to patients it has already seen. A fixed number of rounds avoids that. With no subsampling left, the model has no random component, so unlike the forest it should have no seed variation; that is verified below.

**What might differ from the other two (predictions to check)**

1. It can fit subtler patterns than the forest, so it might do better on the beats both others miss (atrial, fusion, junctional).
2. It shares the forest's risk of fitting patient-specific quirks, and later trees fit whatever is left over - so performance on unseen patients could get *worse* with more rounds. Checked below by scoring held-out patients after every number of rounds.
3. A log-loss objective gives sharp probabilities close to 0 or 1, which may be overconfident on new patients. Checked with a calibration plot.
4. Fast and deterministic - no seed to lose.

**Ground rules** - identical to notebook 10: same data and folds, same evaluation code and metrics, 0.5 threshold for the threshold-based numbers, untuned defaults, and a patient-level bootstrap to judge whether differences are real.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

from src.evaluation import (annotated_rhythm, flagged_rate_by_symbol, metrics_by_group, out_of_fold_probabilities,
                            patient_bootstrap, permutation_importance_by_fold, summarize)
from src.feature_extraction import MODEL_FEATURES, add_record_relative_features, load_beat_table
from src.models import make_gradient_boosting, make_logistic_regression, make_random_forest
from src.splitting import apply_split, load_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

table = add_record_relative_features(load_beat_table()).sort_values(["record", "r_peak_sample"]).reset_index(drop=True)
data = apply_split(table, load_split())
dev = data[data.split == "train"].copy()
del table, data                                   # the test rows are gone from this notebook
print(f"development set: {len(dev)} beats, {dev.record.nunique()} patients, {100 * dev.is_abnormal.mean():.1f}% abnormal")

## Training and prediction

Out-of-fold probabilities for all three models (the logistic regression and forest are deterministic, so recomputing them here reproduces notebooks 09 and 10), plus the boosting model with `class_weight="balanced"` as the same single labelled variant the other two had.

In [ ]:
lr = out_of_fold_probabilities(make_logistic_regression, dev)
rf = out_of_fold_probabilities(make_random_forest, dev)                       # seed 42, as in notebook 10
gb = out_of_fold_probabilities(make_gradient_boosting, dev)
gb_balanced = out_of_fold_probabilities(lambda: make_gradient_boosting(class_weight="balanced"), dev)

rows = {
    "logistic regression": summarize(dev.is_abnormal, lr),
    "random forest": summarize(dev.is_abnormal, rf),
    "gradient boosting": summarize(dev.is_abnormal, gb),
    "gradient boosting, balanced": summarize(dev.is_abnormal, gb_balanced),
}
comparison = pd.DataFrame(rows).T[["accuracy", "precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float).round(3)
print(f"No-skill PR-AUC (prevalence): {dev.is_abnormal.mean():.3f}   |   'always Normal' accuracy: {1 - dev.is_abnormal.mean():.3f}")
comparison

In [ ]:
# does the seed matter? (it should not: nothing random is left in this configuration)
for seed in [0, 1, 2]:
    other = out_of_fold_probabilities(lambda seed=seed: make_gradient_boosting(random_state=seed), dev)
    print(f"seed {seed}: largest difference in any out-of-fold probability vs seed 42 = {float(np.abs(other - gb).max())}")

### Is the difference real? Patient-level bootstrap against both models

1,000 resamples of the 12 patients (with replacement); a positive difference means boosting is ahead.

In [ ]:
rows = []
for other_name, other in [("logistic regression", lr), ("random forest", rf)]:
    boot = patient_bootstrap(dev, other, gb, n_boot=1000, seed=0)
    for metric, label in [("roc_auc", "ROC-AUC"), ("pr_auc", "PR-AUC"), ("f1", "F1 at threshold 0.5")]:
        diff = boot[metric + "_b"] - boot[metric + "_a"]
        rows.append((f"boosting minus {other_name}", label, diff.mean(), *np.percentile(diff, [2.5, 97.5]), (diff > 0).mean()))
pd.DataFrame(rows, columns=["comparison", "metric", "mean difference", "2.5th percentile", "97.5th percentile",
                            "share of resamples where boosting is better"]).round(3)

Reading the pooled table and the two bootstrap comparisons:

- **Boosting lands between the other two - it is not a new winner.** ROC-AUC 0.820 (logistic regression 0.762, forest 0.904), PR-AUC 0.689 (0.680, 0.725), F1 at 0.5 of 0.614 (0.645, 0.612), false alarms 6.4% (4.0%, 6.2%), recall 0.60 (0.58, 0.59).
- **Against the logistic regression:** better ranking by ROC-AUC (+0.053, 95% interval 0.001 to 0.120, ahead in 98% of resamples - once more only just excluding zero), no distinguishable PR-AUC difference (+0.009, interval -0.039 to +0.071), and a lower F1 at the default threshold (-0.030, boosting ahead in only 4% of resamples).
- **Against the forest:** every interval includes zero (ROC-AUC -0.074, PR-AUC -0.028, F1 +0.002). On these 12 patients the two are statistically indistinguishable, with the forest nominally ahead on ranking.
- **Deterministic and fast:** probabilities are identical for every seed (difference 0.0), and the whole four-fold cross-validation takes about a second.
- **Balanced class weights** (F1 0.605, ROC-AUC 0.849) shift the operating point slightly and do not help; the default stays.

## Checking the predictions: rounds, and calibration

**Rounds.** The boosting model was fixed at 100 rounds in advance. Because a boosted model can be read after any number of rounds, one fit per fold shows how held-out-patient performance changes as trees are added. This is a *diagnostic* of whether 100 is a sensible, non-critical setting - not a search: the model stays at 100 unless the curve shows something egregious.

**Calibration.** Beats are sorted by predicted probability into 8 equal-sized groups; in a calibrated model the average predicted probability equals the observed share of abnormal beats in each group.

In [ ]:
rounds = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300]
position = {index: i for i, index in enumerate(dev.index)}
staged = {n: np.full(len(dev), np.nan) for n in rounds}
for fold in sorted(dev.cv_fold.unique()):
    fit, held = dev[dev.cv_fold != fold], dev[dev.cv_fold == fold]
    model = make_gradient_boosting(max_iter=max(rounds)).fit(fit[MODEL_FEATURES], fit.is_abnormal)
    held_rows = [position[i] for i in held.index]
    for n, proba in enumerate(model.staged_predict_proba(held[MODEL_FEATURES]), start=1):
        if n in staged:
            staged[n][held_rows] = proba[:, 1]

y = dev.is_abnormal.to_numpy()
by_rounds = pd.DataFrame({n: {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p)}
                          for n, p in staged.items()}).T
by_rounds.index.name = "boosting rounds"
assert abs(by_rounds.loc[100, "PR-AUC"] - summarize(dev.is_abnormal, gb)["pr_auc"]) < 1e-9   # same model as the main run
by_rounds.round(3)

In [ ]:
calibration = {}
for name, proba in [("logistic regression", lr), ("random forest", rf), ("gradient boosting", gb)]:
    observed, predicted = calibration_curve(dev.is_abnormal, proba, n_bins=8, strategy="quantile")
    calibration[name] = (predicted, observed)

print("Average predicted probability -> observed share of abnormal beats, for 8 equal-sized groups (lowest to highest)")
pd.concat({name: pd.DataFrame({"predicted": p, "observed": o}).round(2).T for name, (p, o) in calibration.items()})

In [ ]:
prevalence = dev.is_abnormal.mean()
colors = {"logistic regression": "tab:blue", "random forest": "tab:orange", "gradient boosting": "tab:green"}
probabilities = {"logistic regression": lr, "random forest": rf, "gradient boosting": gb}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

ax = axes[0][0]
for name, proba in probabilities.items():
    precision, recall, _ = precision_recall_curve(dev.is_abnormal, proba)
    s = summarize(dev.is_abnormal, proba)
    ax.plot(recall, precision, color=colors[name], label=f"{name} (PR-AUC {s['pr_auc']:.2f})")
    ax.scatter(s["recall"], s["precision"], color=colors[name], edgecolor="black", zorder=3)
ax.axhline(prevalence, color="grey", linestyle="--", label=f"no skill ({prevalence:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-recall (dots = threshold 0.5)")
ax.legend(loc="upper right")

ax = axes[0][1]
per_patient = {name: metrics_by_group(dev, proba, "record") for name, proba in probabilities.items()}
order = list(per_patient["logistic regression"].index)
y_pos = np.arange(len(order))
for k, (name, part) in enumerate(per_patient.items()):
    ax.barh(y_pos + (k - 1) * 0.27, part.loc[order, "pr_auc"].astype(float), height=0.27, color=colors[name], label=name)
info = per_patient["logistic regression"]
ax.set_yticks(y_pos, [f"{r} ({100 * info.loc[r, 'abnormal'] / info.loc[r, 'beats']:.0f}% abn)" for r in order])
ax.invert_yaxis()
ax.set_xlabel("PR-AUC within the patient")
ax.set_title("Per patient")
ax.legend(loc="lower right", fontsize=8)

ax = axes[1][0]
ax.plot(by_rounds.index, by_rounds["PR-AUC"], marker="o", color="tab:green", label="PR-AUC")
ax.plot(by_rounds.index, by_rounds["ROC-AUC"], marker="o", color="tab:purple", label="ROC-AUC")
ax.axvline(100, color="grey", linestyle="--", label="fixed setting (100)")
ax.set_xscale("log")
ax.set_xlabel("Boosting rounds (log scale)")
ax.set_ylabel("Score on held-out patients (out-of-fold)")
ax.set_title("More rounds are not automatically better on unseen patients")
ax.legend(loc="lower right")

ax = axes[1][1]
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="perfectly calibrated")
for name, (predicted, observed) in calibration.items():
    ax.plot(predicted, observed, marker="o", color=colors[name], label=name)
ax.set_xlabel("Average predicted probability")
ax.set_ylabel("Observed share of abnormal beats")
ax.set_title("Calibration on held-out patients (8 equal-sized groups)")
ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig("../results/figures/23_boosting_comparison.png", dpi=120)
plt.show()

**Checking the predictions**

- **Rounds (prediction 2): mildly confirmed.** On held-out patients PR-AUC climbs until about 50-75 rounds (0.694 at 75) and then declines slowly (0.689 at 100, 0.674 at 300); ROC-AUC peaks at 50 rounds (0.848) and falls to 0.794 by 300. Beyond a point, extra trees fit quirks of the training patients rather than anything that transfers. The effect is small, and the pre-set 100 rounds is within 0.005 PR-AUC of the best value, so 100 is a reasonable and non-critical setting and stays. (Picking 50 or 75 from this curve would be exactly the kind of tuning we agreed not to do.)
- **Calibration (prediction 3): confirmed, and not unique to boosting.** In the highest-probability group boosting predicts 0.98 while 72% of those beats are truly abnormal; the forest predicts 0.89 (72% observed) and the logistic regression 0.91 (72%). In the lowest groups, where boosting and the logistic regression predict essentially 0, between 3% and 17% of beats are abnormal. None of the three is calibrated on unseen patients: what a probability *means* is learned on the training patients and does not carry over. That is one more reason to compare models with threshold-free metrics and to treat any 0.5 cut-off as unvalidated.
- **In the precision-recall panel** boosting tracks the logistic regression through the middle of the range and is slightly better in the high-recall tail (precision about 0.22-0.35 against 0.17-0.2 for recall between 0.7 and 0.95). The forest's distinctive plateau near 0.47-0.49 is not shared by boosting.

## Where do the three models differ?

Per cross-validation fold, per patient and per beat type; false alarms by annotated rhythm (rhythm labels are for analysis only, never inputs). All out-of-fold at threshold 0.5.

In [ ]:
def three_way(group_column, columns):
    parts = {name: metrics_by_group(dev, proba, group_column)[list(columns)] for name, proba in
             [("LR", lr), ("RF", rf), ("GB", gb)]}
    return pd.concat(parts, axis=1).astype(float).round(3)


print("Per cross-validation fold")
three_way("cv_fold", ["recall", "false_alarm_rate", "f1", "pr_auc"])

In [ ]:
print("Per patient")
info = metrics_by_group(dev, lr, "record")[["beats", "abnormal"]].astype(int)
pd.concat([pd.concat({"": info}, axis=1), three_way("record", ["recall", "false_alarm_rate", "pr_auc"])], axis=1)

In [ ]:
by_symbol = pd.DataFrame({
    "label": flagged_rate_by_symbol(dev, lr).label,
    "beats": flagged_rate_by_symbol(dev, lr).beats,
    "LR % flagged": flagged_rate_by_symbol(dev, lr).pct_flagged_abnormal,
    "RF % flagged": flagged_rate_by_symbol(dev, rf).pct_flagged_abnormal,
    "GB % flagged": flagged_rate_by_symbol(dev, gb).pct_flagged_abnormal,
})
print("Share of each beat type flagged as abnormal: recall for the abnormal types, false-alarm rate for the normal ones")
by_symbol

In [ ]:
rhythm = annotated_rhythm(dev)
normal_beats = dev[dev.label == "Normal"].assign(rhythm=rhythm, lr=(lr >= 0.5).astype(int), rf=(rf >= 0.5).astype(int),
                                                  gb=(gb >= 0.5).astype(int))
by_rhythm = normal_beats.groupby("rhythm").agg(normal_beats=("gb", "size"), lr=("lr", "mean"), rf=("rf", "mean"), gb=("gb", "mean"))
by_rhythm[["lr", "rf", "gb"]] = (100 * by_rhythm[["lr", "rf", "gb"]]).round(1)
by_rhythm.columns = ["normal_beats", "LR false-alarm %", "RF false-alarm %", "GB false-alarm %"]
by_rhythm.sort_values("normal_beats", ascending=False)

Where the three differ (out-of-fold, threshold 0.5):

- **Per fold**, boosting has the best PR-AUC in folds 0 and 3 (0.78 and 0.95) and by far the worst in fold 1 (0.38, against 0.52 for the logistic regression and 0.61 for the forest) - the fold that holds record 232.
- **Per patient**, boosting is best on 213 (PR-AUC 0.76; recall 0.70 against 0.61 for the forest and 0.46 for the logistic regression) and on 220 (PR-AUC 0.98, though its recall of 0.31 is below the forest's 0.54), and is worst or near-worst on 232 (0.62), 202 (0.14) and 124 (0.54). The same pattern as the forest: it helps where a combination of features separates the classes, and hurts on patients that sit outside the training range or have irregular rhythm.
- **By beat type:** ventricular beats 99.1% (all three about 99%); atrial beats only 7.6% flagged - the lowest of the three (10.4% logistic regression, 8.6% forest); fusion beats 49.1% - the best (35.0% forest, 19.8% logistic regression); junctional 1.2% for all. False alarms on Normal beats: `N` 7.5% (the highest; logistic regression 4.8%) and `R` 2.1%.
- **Irregular rhythm is still not solved:** false alarms in atrial fibrillation 24.4% (forest 25.2%, logistic regression 17.4%), flutter 98.1%, sinus bradycardia 10.3% (logistic regression 0%), and bigeminy 5.1% - the highest of the three there (logistic regression 1.5%, forest 2.8%).

## What does each tree model rely on?

Permutation importance on unseen patients (the drop in PR-AUC when one feature is shuffled in the held-out fold, averaged over the four folds), for the forest and for boosting. The logistic regression is left out: its weights are not comparable because of the collinear features (notebook 09).

In [ ]:
importance = pd.DataFrame({
    "random forest": permutation_importance_by_fold(make_random_forest, dev).mean(axis=1),
    "gradient boosting": permutation_importance_by_fold(make_gradient_boosting, dev).mean(axis=1),
}).sort_values("gradient boosting", ascending=False)
importance.round(3)

The two tree models rely on different things. The forest leans on absolute amplitude and beat width (`amp_max` 0.088, `qrs_fwhm_ms` 0.081, `amp_min` 0.079); boosting spreads its reliance more thinly (`amp_max` 0.037, `amp_std_rel` 0.028, `qrs_fwhm_ms` 0.021) and uses the timing features somewhat more (`rr_pre_rel` 0.019, `rr_pre_s` 0.017), while `rr_ratio`, which the forest used (0.023), contributes almost nothing to it (0.002).

What they agree on: `rr_post_rel`, `amp_mean` and `rr_post_s` have zero or *negative* importance for both - shuffling them does not hurt on unseen patients. That makes them candidates for removal, but pruning features on this evidence is a design decision to take deliberately (and re-run for every model), not something to do quietly inside a model comparison.

## Summary

**Model:** `HistGradientBoostingClassifier`, scikit-learn defaults (learning rate 0.1, up to 31 leaves, 100 rounds), early stopping off, raw features; deterministic. Evaluated by the same grouped cross-validation on the 12 development patients as the other two. The test set is untouched.

| pooled, out-of-fold | precision | recall | F1 | false alarms | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|---|
| logistic regression | 0.73 | 0.58 | 0.65 | 4.0% | 0.76 | 0.68 |
| random forest | 0.64 | 0.59 | 0.61 | 6.2% | 0.90 | 0.73 |
| gradient boosting | 0.63 | 0.60 | 0.61 | 6.4% | 0.82 | 0.69 |

**Verdict: none of the three is clearly best.** On PR-AUC - the threshold-free metric that suits an imbalanced problem - every difference is within patient-level noise. ROC-AUC favours the two tree models over the logistic regression; F1 at the default threshold favours the logistic regression by about 0.03. Boosting adds nothing over the forest: statistically indistinguishable, but deterministic and quick.

**What the three models say together**

1. **Model choice matters less than the features.** All three catch ventricular beats (about 99%), all three miss atrial beats (8-10% flagged), and all three flag atrial fibrillation and flutter heavily.
2. **Between-patient differences dwarf between-model differences:** each model's PR-AUC ranges from about 0.06 to 1.0 across patients.
3. The tree models help on specific patients (213's fusion beats, 220's atrial beats) and hurt on others (232, 202).
4. **No probability is calibrated on unseen patients**, so the 0.5 threshold is unvalidated for every model.

## Where this leaves the project

The model comparison on the development set is complete. Before the test set is looked at, two things need deciding - deliberately, and in advance:

1. **The selection rule:** which model goes to the test set, and by what criterion. A defensible option is "highest cross-validated PR-AUC, but prefer the simplest model whenever the difference is within patient-level noise", which here would select the logistic regression.
2. **Whether to attempt the one feature idea that all three models' errors point to** (a rhythm-context feature such as the variability of recent RR intervals) before that final choice - since it would change the features every model uses, it has to come first, not after.